# §63 — DoTA train captions: **pilot on 30 clips** (Qwen2.5-VL, prompt with DoTA's labels)

Rule: `core/docs/final/KIM_RUN.md` **§63**. Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first. **A data artifact, not an arm**: nothing trains, and no caption is used anywhere until you decide after this pilot.

What it does: 30 recovered DoTA train clips (one per accident type + 20 at random, seed 2024) → 16 frames over `[onset − 2 s, min(onset + 3 s, anomaly_end)]` at 700 × 392 → Qwen2.5-VL with a dashcam prompt that **carries DoTA's own label** (accident type + whether the ego vehicle is involved) → one sentence per clip. You rate the 30 on LaGoVAD's Tab. B 1–5 scale.

| step | what | GPU | estimate |
|---|---|---|---|
| 1-3 | mount, paths, annotation zip, deps (+ bitsandbytes) | no | 3-4 min |
| 4 | preflight: 3,020 clips, the 30 pilot archives on Drive | no | 1 min |
| 5 | pytest `test_dota_caption.py` + `test_nexar_caption.py` — **raises if red** | no | < 1 min |
| 6 | pin the model revision (reuses Nexar's pin if the model is the same) | no | seconds |
| 7 | **pilot: caption 30 clips** (resumable) | yes | ~20-40 min incl. first download (72B ≈ 145 GB to VM disk) |
| 8 | sheet + 30 contact sheets | no | 1 min |
| 9 | **you** fill `score` 1-5 (+ optional `type_ok` / `roles_ok` = y/n) in `qc_ratings.csv`, then `qc-score` → **paste to Claude** | no | your time |

**Runtime:** 80 GB GPU (A100-80GB / H100) for the 72B in NF4. On a 40 GB A100 set `MODEL = 'Qwen/Qwen2.5-VL-32B-Instruct'` in step 2 (its own folder; never mixed with 72B records). Thesis is only read; everything new goes to `Thesis-Final/outputs/dota_captions/`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Paths, code, annotation zip

In [ ]:
import hashlib
import json
import os
import subprocess
import sys
import time
import urllib.request
from collections import Counter
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code, DoTA train frames, every output of this notebook
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'
os.environ['HF_HOME'] = '/content/hf'             # model weights on VM disk, not Drive FUSE

REPO = Path(f'{DRIVE_F}/kat-vad')
assert (REPO / 'core/tools/dota_caption.py').is_file(), 'this upload has no core/tools/dota_caption.py -- re-upload kat-vad/'
assert '## 63. ' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), 'KIM_RUN.md has no §63 -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
MODEL   = constants.NEXAR_CAPTION_MODEL            # Qwen2.5-VL-72B-Instruct; 'Qwen/Qwen2.5-VL-32B-Instruct' on a 40 GB A100
QUANT   = constants.NEXAR_CAPTION_QUANT            # 'nf4'
ARCHIVE = Path(DRIVE_F) / 'data' / constants.DOTA_TRAIN_FRAMES_DIRNAME     # {dota_id}.zip (§52)
OUT     = constants.OUTPUT_ROOT / constants.DOTA_CAPTION_OUTPUT_DIRNAME
NEXAR_PIN = constants.OUTPUT_ROOT / constants.NEXAR_CAPTION_OUTPUT_DIRNAME / f"revision_{MODEL.split('/')[-1]}.json"
LOCAL   = Path('/content/dota_captions')
for p in (OUT, LOCAL):
    p.mkdir(parents=True, exist_ok=True)

ANN = LOCAL / 'DoTA_annotations.zip'              # DoTA's official annotation zip (10 MB)
if not ANN.exists():
    urllib.request.urlretrieve('https://github.com/MoonBlvd/Detection-of-Traffic-Anomaly/raw/master/dataset/DoTA_annotations.zip', ANN)
frozen = json.loads((constants.DOTA_TRAIN_SPLITS_DIR / constants.DOTA_TRAIN_MAP_FILENAME).read_text())
assert hashlib.sha256(ANN.read_bytes()).hexdigest() == frozen['sources']['annotations_sha256'], 'annotation zip is not the one the map was built on'
print('paths OK |', MODEL, QUANT, '| out', OUT)

## 3. Dependencies

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av "bitsandbytes>=0.45"
python -c "import av, bitsandbytes, torch, transformers; print('deps OK', transformers.__version__, 'bnb', bitsandbytes.__version__, 'torch', torch.__version__)"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- step 7 needs one'
df -h /content | tail -1

## 4. Preflight (CPU)

In [ ]:
import shutil  # noqa: E402

import torch  # noqa: E402

from core.tools import dota_caption as dc  # noqa: E402
from core.tools import nexar_caption as nc  # noqa: E402
from core.tools.dota_train_extract import train_clips  # noqa: E402

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')


def run(cmd):
    '''Stream a child's output here; raise with its tail on failure.'''
    tail = []
    with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        for line in proc.stdout:
            tail = (tail + [line])[-40:]
            print(line, end='')
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


CLIPS = train_clips(constants.DOTA_TRAIN_SPLITS_DIR / constants.DOTA_TRAIN_MAP_FILENAME)
assert len(CLIPS) == 3020, len(CLIPS)
ANNS = dc.load_annotations(ANN, sorted(CLIPS))                  # raises on an unknown accident type
for vid, a in ANNS.items():
    dc.caption_window(a)                                         # raises if a clip cannot give 16 frames
PILOT = dc.pilot_sample(ANNS)
missing = [v for v in PILOT if not (ARCHIVE / f'{v}.zip').is_file()]
assert not missing, f'{len(missing)} pilot archives missing under {ARCHIVE}: {missing[:5]}'
print(f'preflight OK: {len(CLIPS)} clips, {len(PILOT)} pilot archives on Drive')
print('pilot types:', dict(Counter(ANNS[v]['accident_name'] for v in PILOT)))
print('pilot ego-involved:', sum(ANNS[v]['ego_involve'] for v in PILOT), '/', len(PILOT))
print('\nexample prompt:\n' + dc.prompt_for(ANNS[PILOT[0]]))

## 5. Tests (CPU) — **raises if anything fails: stop and paste the output**

In [ ]:
run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'core/tests/test_dota_caption.py', 'core/tests/test_nexar_caption.py'])

## 6. Pin the model revision once (C4)

If Nexar's captions already pinned this model, the same commit is reused (same weights as §29.3); otherwise the Hub's current commit is written to `OUT/revision_<model>.json`.

In [ ]:
from huggingface_hub import HfApi  # noqa: E402

REV_FILE = OUT / f"revision_{MODEL.split('/')[-1]}.json"
if not REV_FILE.exists():
    pin = json.loads(NEXAR_PIN.read_text()) if NEXAR_PIN.exists() else {'model': MODEL, 'revision': HfApi().model_info(MODEL).sha}
    REV_FILE.write_text(json.dumps(pin, indent=2))
REV = json.loads(REV_FILE.read_text())
assert REV['model'] == MODEL, REV
REVISION = REV['revision']
print(f'{MODEL} @ {REVISION}', '(from Nexar pin)' if NEXAR_PIN.exists() else '')

## 7. Pilot: caption the 30 clips (GPU, resumable)

A disconnect is harmless: re-run steps 1-6 and this cell; finished clips are skipped.

In [ ]:
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), 'needs a bf16 GPU (A100 / H100)'
gb = torch.cuda.get_device_properties(0).total_memory / 2**30
need = 70 if '72B' in MODEL else 35
assert gb >= need, f'{gb:.0f} GB GPU: {MODEL} in NF4 needs >= {need} GB (set MODEL to the 32B on a 40 GB A100)'
free = shutil.disk_usage('/content').free / 2**30
weights = 150 if '72B' in MODEL else 70
cached = Path(os.environ['HF_HOME']) / 'hub' / f"models--{MODEL.replace('/', '--')}"
assert cached.exists() or free > weights + 10, f'free {free:.0f} GB on /content; the weights need ~{weights} GB'
print(f'GPU {torch.cuda.get_device_name(0)} {gb:.0f} GB | disk free {free:.0f} GB')
t0 = time.time()
run([sys.executable, '-m', 'core.tools.dota_caption', 'pilot', '--archive-dir', ARCHIVE, '--annotations', ANN,
     '--out-dir', OUT, '--revision', REVISION, '--model', MODEL, '--quant', QUANT])
PILOT_DIR = OUT / dc.PILOT_DIRNAME / dc.pilot_dirname(MODEL)
summary = json.loads((PILOT_DIR / dc.PILOT_SUMMARY_FILENAME).read_text())
assert summary['clips'] == len(PILOT) and summary['ids'] == PILOT and summary['revision'] == REVISION, summary
print(f'{summary["clips"]} captions in {(time.time() - t0) / 60:.0f} min')
print(json.dumps({k: v for k, v in summary.items() if k not in ('ids', 'prompt_sha1')}, indent=1))

## 8. Sheet: 30 clips, each with the 16 frames the model saw

**How to rate** (Tab. B 1–5, judged against the frames, not against the label): 5 = accurate and complete; 4 = minor slips; 3 = main event right but notable omissions / errors; 2 = largely wrong; 1 = nothing to do with the video. Optional: `type_ok` = y if the caption matches the annotated type, `roles_ok` = y if the road users and who does what are right. Note in a free line anything systematic (e.g. "says the ego crashes when it only watches").

In [ ]:
from IPython.display import Image, Markdown, display  # noqa: E402

QC = PILOT_DIR / nc.QC_DIRNAME
display(Markdown((QC / nc.QC_SHEET_FILENAME).read_text()))
for png in sorted(QC.glob('*.png')):
    print(png.stem)
    display(Image(filename=str(png), width=900))
print(f'Now open {QC / nc.QC_RATINGS_FILENAME} on Drive, fill score (1-5) for all 30 rows (optional type_ok / roles_ok = y/n), save, then run step 9.')

## 9. QC score — **after you filled `qc_ratings.csv`; paste the printed block (and your notes) to Claude**

In [ ]:
run([sys.executable, '-m', 'core.tools.dota_caption', 'qc-score', '--pilot-dir', PILOT_DIR])
print(json.dumps(json.loads((QC / nc.QC_SUMMARY_FILENAME).read_text()), indent=1))
print((QC / nc.QC_SHEET_FILENAME).read_text())
drive.flush_and_unmount()          # C10: every pending Drive write lands before the VM goes